# Q-learning: one inspected table update

Companion to HTB Academy Module 290, Section 14. A Q-table row is a state; a column is an action; the cell Q(s,a) estimates future reward after taking that action in that state. Read s-prime as **the next state**, a-prime as **an available next action**, alpha as **the learning rate**, gamma as **the discount factor**, and epsilon as **the exploration probability**. This notebook reproduces HTB's one-step S1/Right example. It does not claim to train an optimal policy.

In [ ]:
from copy import deepcopy
import random

# Each dictionary is a state row. Its keys are available actions (columns).
q_table = {
    'S1': {'Up': -1.0, 'Down': 0.0, 'Left': -0.5, 'Right': 0.2},
    'S2': {'Up': 0.0, 'Down': 1.0, 'Left': 0.0, 'Right': -0.3},
    'S3': {'Up': 0.5, 'Down': -0.5, 'Left': 1.0, 'Right': 0.0},
    'S4': {'Up': -0.2, 'Down': 0.0, 'Left': -0.3, 'Right': 1.0},
}
before = deepcopy(q_table)  # Save all cells for the one-cell-change check.
state, action, next_state = 'S1', 'Right', 'S2'
reward, alpha, gamma = 0.5, 0.1, 0.9
assert 0 <= alpha <= 1 and 0 <= gamma <= 1
assert action in q_table[state] and q_table[next_state]

# HTB formula: target = r + gamma * max_a-prime Q(s-prime,a-prime).
old_value = q_table[state][action]
best_next = max(q_table[next_state].values())
target = reward + gamma * best_next
# TD error is target minus the old prediction for this specific state-action pair.
td_error = target - old_value
# New Q = old Q + alpha * TD error: move only partway toward target.
new_value = old_value + alpha * td_error
q_table[state][action] = new_value
print(f'old={old_value}; best next={best_next}; target={target}; error={td_error}; new={new_value}')
assert abs(target - 1.4) < 1e-12
assert abs(td_error - 1.2) < 1e-12
assert abs(new_value - 0.32) < 1e-12
assert all(q_table[s][a] == before[s][a] for s in q_table for a in q_table[s] if (s, a) != (state, action))

## Terminal next state

A terminal state ends the episode, so no next action exists. Its future contribution is zero: the target is just the immediate reward. This is a boundary case; it is not part of HTB's nonterminal S1-to-S2 arithmetic.

In [ ]:
terminal_reward = 2.0
terminal_target = terminal_reward + gamma * 0.0
print('terminal target:', terminal_target)
assert terminal_target == 2.0

## Epsilon-greedy action selection

Read epsilon (ε) as **the probability of choosing the random-action branch**. With probability one minus epsilon, choose a maximum-Q action. A random branch might pick that same greedy action. Ties in maximum Q are resolved by the insertion order of the row in this teaching implementation; production code should decide and document a tie policy.

In [ ]:
def epsilon_greedy(row, epsilon, rng):
    """Choose among one state's available actions, with no environment side effects."""
    if not row or not 0 <= epsilon <= 1:
        raise ValueError('Need available actions and epsilon in [0,1]')
    actions = list(row)  # Keys are the available action names.
    if rng.random() < epsilon:
        return rng.choice(actions), 'explore'
    return max(actions, key=lambda a: row[a]), 'exploit'

rng = random.Random(14)  # Fixed seed makes this example reproducible.
greedy_action, greedy_mode = epsilon_greedy(q_table['S2'], 0.0, rng)
random_action, random_mode = epsilon_greedy(q_table['S2'], 1.0, rng)
print('epsilon=0:', greedy_action, greedy_mode)
print('epsilon=1:', random_action, random_mode)
assert (greedy_action, greedy_mode) == ('Down', 'exploit')
assert random_mode == 'explore' and random_action in q_table['S2']

The table contains estimates, not certainties. Repeated safe experiences and a suitable state/reward design are required before a policy can be evaluated; one numerical update and two selection checks cannot establish convergence.